<a href="https://colab.research.google.com/github/192565027simats/CSA6102/blob/main/EXP30-Linux%20SSH%20Auth%20Log%20Suspicious%20Login%20Detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import re

# Regular expression to parse SSH authentication log entries
AUTH_LINE_RE = re.compile(
    r"(?P<result>Accepted|Failed) password for (?P<user>\S+) "
    r"from (?P<ip>[\d.]+) port (?P<port>\d+)"
)


def parse_auth_log(lines):
    """
    Parse raw auth.log lines into structured dictionaries.
    """
    entries = []

    for line in lines:
        match = AUTH_LINE_RE.search(line)
        if match:
            entries.append({
                "result": match.group("result"),
                "user": match.group("user"),
                "ip": match.group("ip"),
                "port": int(match.group("port")),
                "raw": line,
            })

    return entries


def flag_suspicious_logins(entries, trusted_ips):
    """
    Flag successful logins from untrusted IPs,
    especially successful root logins.
    """
    flagged = []

    for entry in entries:
        if entry["result"] == "Accepted" and entry["ip"] not in trusted_ips:
            severity = "HIGH" if entry["user"] == "root" else "MEDIUM"

            flagged.append({
                **entry,
                "severity": severity,
            })

    return flagged


# ---------------------- Test Case ----------------------

def test_experiment3():
    log_lines = [
        "Jan 15 08:00:01 server sshd[1001]: Accepted password for deploy from 10.0.0.5 port 51100 ssh2",
        "Jan 15 03:12:01 server sshd[1233]: Failed password for root from 198.51.100.23 port 51320 ssh2",
        "Jan 15 03:12:05 server sshd[1234]: Accepted password for root from 198.51.100.23 port 51322 ssh2",
    ]

    trusted_ips = {"10.0.0.5"}

    # Parse log entries
    entries = parse_auth_log(log_lines)

    # Verify parsing
    assert len(entries) == 3

    # Detect suspicious successful logins
    flagged = flag_suspicious_logins(entries, trusted_ips)

    # Verify detection
    assert len(flagged) == 1
    assert flagged[0]["user"] == "root"
    assert flagged[0]["ip"] == "198.51.100.23"
    assert flagged[0]["severity"] == "HIGH"

    print("All test cases passed.")

    print("\nFlagged Suspicious Logins:")
    for item in flagged:
        print(item)


# Run the test
test_experiment3()

All test cases passed.

Flagged Suspicious Logins:
{'result': 'Accepted', 'user': 'root', 'ip': '198.51.100.23', 'port': 51322, 'raw': 'Jan 15 03:12:05 server sshd[1234]: Accepted password for root from 198.51.100.23 port 51322 ssh2', 'severity': 'HIGH'}
